## 🎬 Movie Success Prediction Using Classification Algorithms


## Objectives

- Understand the dataset
- Perform Exploratory Data Analysis (EDA)
- Handle missing values and outliers
- Perform feature engineering
- Train multiple machine learning models
- Compare model performance
- Select the best model
- Save the model
- Deploy using Streamlit

In [1]:
# Imports & Settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings("ignore")
print("Imports Loaded Successfully")

Imports Loaded Successfully


In [2]:
# Load Meta Data
data = pd.read_csv(r"D:\Capston_Project\movie_metadata.csv")
data.head()

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000
1,Color,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0
2,Color,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7.1,NaN,0


In [3]:
# Configuration
INPUT_FILE = "movie_metadata.csv"
OUTPUT_FILE = "cleaned_movie_data.csv"

In [4]:
# Target Column Setup
TARGET_COLUMN = "imdb_score"
TARGET_LABEL = "imdb_binned"
IMDB_BINS = [1, 3, 6, 10]
IMDB_LABELS = ["FLOP", "AVG", "HIT"]

In [5]:
# Unnecessary Columns to Remove
DROP_COLUMNS = ["movie_title", "movie_imdb_link"]

In [6]:
# Imputation Strategies for Remaining General Columns
NUMERICAL_IMPUTATION = "median"
CATEGORICAL_IMPUTATION = "most_frequent"

print("Configuration Loaded Successfully.")

Configuration Loaded Successfully.


# Data Cleaning & Processing

In [7]:
# Load Dataset
df = pd.read_csv(INPUT_FILE)
df_original = df.copy()

print(f"Dataset loaded successfully from '{INPUT_FILE}'")
print(f"Initial Shape: {df.shape}")

Dataset loaded successfully from 'movie_metadata.csv'
Initial Shape: (5043, 28)


In [8]:
# Remove Duplicates
duplicate_count = df.duplicated().sum()
if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Removed {duplicate_count} duplicate row(s).")
else:
    print("No duplicate rows found.")
print(f"Current Shape: {df.shape}")

Removed 45 duplicate row(s).
Current Shape: (4998, 28)


In [9]:
# Drop Unnecessary Columns
cols_to_drop = [col for col in DROP_COLUMNS if col in df.columns]
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped columns: {cols_to_drop}")
else:
    print("No specified columns found to drop.")
print(f"Current Shape: {df.shape}")

Dropped columns: ['movie_title', 'movie_imdb_link']
Current Shape: (4998, 26)


In [10]:
# Specific Column Imputation & Feature Engineering
# Fill missing values
df['budget'] = df['budget'].fillna(df['budget'].median())
df['gross'] = df['gross'].fillna(df['gross'].median())
df['director_facebook_likes'] = df['director_facebook_likes'].fillna(0)
df['actor_1_facebook_likes'] = df['actor_1_facebook_likes'].fillna(0)
df['actor_2_facebook_likes'] = df['actor_2_facebook_likes'].fillna(0)
df['actor_3_facebook_likes'] = df['actor_3_facebook_likes'].fillna(0)
df['cast_total_facebook_likes'] = df['cast_total_facebook_likes'].fillna(0)
df['num_critic_for_reviews'] = df['num_critic_for_reviews'].fillna(0)
df['num_user_for_reviews'] = df['num_user_for_reviews'].fillna(0)
df['num_voted_users'] = df['num_voted_users'].fillna(0)
df['title_year'] = df['title_year'].fillna(df['title_year'].median())

# Binary Feature Creation
df['is_color'] = (df['color'].fillna('') == 'Color').astype(int)

In [11]:
# Financial & Interaction Features
df['log_budget'] = np.log1p(np.maximum(df['budget'], 0))
df['log_gross'] = np.log1p(np.maximum(df['gross'], 0))
df['roi'] = (df['gross'] - df['budget']) / (df['budget'] + 1000)
df['review_ratio'] = df['num_user_for_reviews'] / (df['num_critic_for_reviews'] + 1)
df['votes_per_review'] = df['num_voted_users'] / (df['num_user_for_reviews'] + 1)
df['actor_lead_share'] = df['actor_1_facebook_likes'] / (df['cast_total_facebook_likes'] + 1)
df['director_share'] = df['director_facebook_likes'] / (df['cast_total_facebook_likes'] + 1)
df['movie_age'] = 2026 - df['title_year']

print("Domain-specific missing values handled and engineered features generated.")

Domain-specific missing values handled and engineered features generated.


In [12]:
# Handle Remaining Missing Values
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"All Numerical columns:,{num_cols}")
print(f"All Categorical columns:,{cat_cols}")

All Numerical columns:,['num_critic_for_reviews', 'duration', 'director_facebook_likes', 'actor_3_facebook_likes', 'actor_1_facebook_likes', 'gross', 'num_voted_users', 'cast_total_facebook_likes', 'facenumber_in_poster', 'num_user_for_reviews', 'budget', 'title_year', 'actor_2_facebook_likes', 'imdb_score', 'aspect_ratio', 'movie_facebook_likes', 'is_color', 'log_budget', 'log_gross', 'roi', 'review_ratio', 'votes_per_review', 'actor_lead_share', 'director_share', 'movie_age']
All Categorical columns:,['color', 'director_name', 'actor_2_name', 'genres', 'actor_1_name', 'actor_3_name', 'plot_keywords', 'language', 'country', 'content_rating']


In [13]:
# Exclude target score from generic imputation
if TARGET_COLUMN in num_cols:
    num_cols.remove(TARGET_COLUMN)
# Impute Remaining Numerical Columns
if num_cols:
    num_imputer = SimpleImputer(strategy=NUMERICAL_IMPUTATION)
    df[num_cols] = num_imputer.fit_transform(df[num_cols])
    print(f"Imputed remaining numerical columns using strategy='{NUMERICAL_IMPUTATION}'")
# Impute Remaining Categorical Columns
if cat_cols:
    cat_imputer = SimpleImputer(strategy=CATEGORICAL_IMPUTATION)
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])
    print(f"Imputed remaining categorical columns using strategy='{CATEGORICAL_IMPUTATION}'")

print(f"Remaining missing values across dataset: {df.isnull().sum().sum()}")

Imputed remaining numerical columns using strategy='median'
Imputed remaining categorical columns using strategy='most_frequent'
Remaining missing values across dataset: 0


In [14]:
# Bin Target Variable

if TARGET_COLUMN in df.columns:
    df.dropna(subset=[TARGET_COLUMN], inplace=True)
    df.reset_index(drop=True, inplace=True)

    df[TARGET_LABEL] = pd.cut(
        df[TARGET_COLUMN],
        bins=IMDB_BINS,
        labels=IMDB_LABELS,
        include_lowest=True
    )

    df.drop(columns=[TARGET_COLUMN], inplace=True)
    print(f"Transformed '{TARGET_COLUMN}' into '{TARGET_LABEL}'")
    print("\nTarget Class Distribution:")
    print(df[TARGET_LABEL].value_counts())

Transformed 'imdb_score' into 'imdb_binned'

Target Class Distribution:
imdb_binned
HIT     3428
AVG     1524
FLOP      46
Name: count, dtype: int64


In [15]:
# 7. Save Cleaned Dataset
df.to_csv(OUTPUT_FILE, index=False)

print(f"Cleaned dataset saved successfully to '{OUTPUT_FILE}'")
print(f"Final Cleaned Shape: {df.shape}")

Cleaned dataset saved successfully to 'cleaned_movie_data.csv'
Final Cleaned Shape: (4998, 35)
